# Создание аугментаций

Можно воспользоваться этим кодом для получения аугментаций, но только в синхронном формате. Для асинхронного формата необходимо запустить скрипт `create_augmentation.py`. В этом скрипте идентичный код.

In [1]:
from autointent import load_dataset
import json
import pandas as pd
import os


def create_subset(dataset_name: str, max_utterances_per_intent: int = 10):
    output_file = f"datasets/{dataset_name.split('/')[-1]}.json"
    output_file_subset = output_file.replace(".json", f"_subset_{max_utterances_per_intent}.json")

    if os.path.isfile(output_file_subset):
        return output_file, output_file_subset
    
    dataset = load_dataset(dataset_name)
    dataset.to_json(output_file)
    
    with open(output_file, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data["train"])
    sampled_df = df.groupby("label").head(max_utterances_per_intent)
    data["train"] = sampled_df.to_dict(orient="records")

    with open(output_file_subset, 'w') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    return output_file, output_file_subset

In [2]:
datasets = [
    "AutoIntent/banking77_ru",
    "AutoIntent/clinc150_ru",
    "AutoIntent/banking77",
    "AutoIntent/snips",
    "AutoIntent/snips_ru"
]

datasets_subset = []

for dataset_name in datasets:
    _, output_file_subset = create_subset(dataset_name, max_utterances_per_intent=10)
    datasets_subset.append(output_file_subset)

In [3]:
from autointent.generation.utterances.evolution.evolver import UtteranceEvolver
from autointent.generation.utterances.generator import Generator
from autointent.generation.utterances.evolution.chat_templates import (
    AbstractEvolution,
    ConcreteEvolution,
    FormalEvolution,
    FunnyEvolution,
    GoofyEvolution,
    InformalEvolution,
    ReasoningEvolution,
)
import os

os.environ["OPENAI_MODEL_NAME"] = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
os.environ["OPENAI_BASE_URL"] = 'http://localhost:8000/v1'
os.environ["OPENAI_API_KEY"] = 'sth'

In [4]:
datasets_subset

['datasets/banking77_ru_subset_10.json',
 'datasets/clinc150_ru_subset_10.json',
 'datasets/banking77_subset_10.json',
 'datasets/snips_subset_10.json',
 'datasets/snips_ru_subset_10.json']

In [ ]:
evolutions = [
        AbstractEvolution(),
        ConcreteEvolution(),
        FormalEvolution(),
        FunnyEvolution(),
        GoofyEvolution(),
        InformalEvolution(),
        ReasoningEvolution()
]
seed = 42
split = "train"
n_evolutions = 10
datasets_subset_aug = []


for dataset_subset in datasets_subset:
    print(dataset_subset)
    dataset = load_dataset(dataset_subset)
    n_before = len(dataset[split])
    output_path = dataset_subset.replace(".json", "_augmented.json") 
    datasets_subset_aug.append(output_path)

    generator = UtteranceEvolver(Generator(), evolutions, seed, async_mode=False)
    _ = generator.augment(dataset, split_name=split, n_evolutions=n_evolutions)

    dataset.to_json(output_path)

# Проведение экспериментов

In [76]:
datasets = [
    "DeepPavlov/banking77_ru",
    "DeepPavlov/clinc150_ru",
    "DeepPavlov/banking77",
    "DeepPavlov/snips",
    "DeepPavlov/snips_ru"
]
max_utterances_per_intent = 10
model_name = "gpt-4o-mini-2024-07-18"

datasets_subset = []
datasets_subset_aug = []

for dataset_name in datasets:
    base_name = dataset_name.split("/")[-1]
    path_to_datasets = "/home/darinka/AutoIntent-experiments/experiments/utterance_augmentation/datasets"
    dataset_base = f"{path_to_datasets}/{base_name}_subset_{max_utterances_per_intent}.json"
    dataset_subset = f"{path_to_datasets}/{model_name.split('/')[-1]}/{base_name}_subset_{max_utterances_per_intent}.json"
    dataset_subset_aug = dataset_subset.replace(".json", "_augmented.json")

    datasets_subset.append(dataset_base)
    datasets_subset_aug.append(dataset_subset_aug)

In [77]:
datasets_subset[0], datasets_subset_aug[0]

('/home/darinka/AutoIntent-experiments/experiments/utterance_augmentation/datasets/banking77_ru_subset_10.json',
 '/home/darinka/AutoIntent-experiments/experiments/utterance_augmentation/datasets/gpt-4o-mini-2024-07-18/banking77_ru_subset_10_augmented.json')

In [78]:
search_space = "heavy_extra" 

In [79]:
from autointent import Pipeline
from autointent.configs import LoggingConfig, EmbedderConfig

log_config = LoggingConfig(
    report_to=["wandb"], clear_ram=True, dump_modules=True)
emb_config = EmbedderConfig(batch_size=16, device="cuda")

In [80]:
from autointent import Dataset
import torch
import wandb


def run_experiment(dataset, n_aug=0, ds_type: str = "json", name: str = "dataset"):
    dataset_base = name.split("/")[-1]
    log_config.run_name = f"{model_name.split('/')[-1]}_{dataset_base}_naug_{n_aug}"

    if ds_type == "json":
        dataset = Dataset.from_json(dataset)
    else:
        dataset = Dataset.from_dict(dataset)

    pipeline_optimizer = Pipeline.from_preset(search_space)
    pipeline_optimizer.set_config(log_config)
    pipeline_optimizer.set_config(emb_config)
    pipeline_optimizer.fit(dataset)
    torch.cuda.empty_cache()
    wandb.finish()

In [ ]:
import json

n_evolutions = 10

for subset, aug in zip(datasets_subset[:1], datasets_subset_aug[:1]):
    print(subset)
    print(aug)
    # run_experiment(subset, 0, "json", subset)
    
    with open(subset, "r") as f:
        dataset = json.load(f)
    
    n = len(dataset["train"])

    with open(aug, "r") as f:
        intents = {}
        dataset_aug = json.load(f)

        for utterance in dataset_aug["train"][n:]:
            label = utterance["label"]
            if label not in intents:
                intents[label] = []
            intents[label].append(utterance)

        for i in range(4, n_evolutions + 1):
            train = dataset_aug["train"][:n]
           
            for label, utterances in intents.items():
                j = 0
                while j < len(utterances):
                    train.extend(utterances[j:j + i])
                    j += i 

                    j += (n_evolutions - i)

            dataset_aug["train"] = train

            run_experiment(dataset_aug, i, "dict", aug)